In [1]:
import hdbscan
import numpy as np
import pandas as pd
import umap
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.preprocessing import normalize as sk_normalize

df = pd.read_csv("../../donnees/jeopardy.csv")

emb_cols = [c for c in df.columns if c.startswith("emb_")]
X = df[emb_cols].values.astype(np.float32)
questions = df["content"].values

print(f"{len(df)} questions, embeddings shape : {X.shape}")

C:\Users\pleroy\Project\anssi-recommandations-cyber\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


110365 questions, embeddings shape : (110365, 1024)


# Normalisation + PCA

In [2]:
X = normalize(X, norm="l2")

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)
print(
    f"Variance expliquée (50 composantes) : {pca.explained_variance_ratio_.sum():.2%}"
)

Variance expliquée (50 composantes) : 50.04%


# Umap

In [3]:
reducer = umap.UMAP(
    n_components=15,
    metric="cosine",
    n_neighbors=30,
    min_dist=0.0,
    random_state=42,
)
X_umap = reducer.fit_transform(X_pca)
print(f"UMAP shape : {X_umap.shape}")

C:\Users\pleroy\Project\anssi-recommandations-cyber\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP shape : (110365, 15)


# HDBSCAN

In [4]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=30,
    metric="euclidean",
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.3,
)
labels = clusterer.fit_predict(X_umap)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_outliers = (labels == -1).sum()
print(f"{n_clusters} clusters, {n_outliers} outliers ({n_outliers / len(labels):.1%})")
print(pd.Series(labels[labels != -1]).value_counts().sort_index().to_string())

7 clusters, 323 outliers (0.3%)
0     1758
1      623
2    17955
3     1275
4    71805
5     3082
6    13544


# Mots-clés par cluster (TF-IDF)

In [5]:
STOP_WORDS_FR = {
    "le",
    "la",
    "les",
    "de",
    "du",
    "des",
    "un",
    "une",
    "en",
    "et",
    "est",
    "sontil",
    "elle",
    "ils",
    "elles",
    "on",
    "je",
    "tu",
    "nous",
    "vous",
    "que",
    "qui",
    "quoi",
    "quel",
    "quelle",
    "quels",
    "quelles",
    "ce",
    "cet",
    "cette",
    "ces",
    "se",
    "sa",
    "son",
    "ses",
    "mon",
    "ma",
    "mes",
    "dans",
    "sur",
    "sous",
    "par",
    "pour",
    "avec",
    "sans",
    "entre",
    "vers",
    "au",
    "aux",
    "ou",
    "où",
    "si",
    "ne",
    "pas",
    "plus",
    "très",
    "bien",
    "mais",
    "donc",
    "car",
    "comme",
    "comment",
    "quand",
    "lors",
    "selon",
    "être",
    "avoir",
    "faire",
    "dit",
    "fait",
    "doit",
    "peut",
    "faut",
    "à",
    "y",
    "en",
    "lui",
    "leur",
    "leurs",
    "tout",
    "tous",
    "toute",
    "toutes",
    "même",
    "aussi",
    "alors",
    "ainsi",
    "dont",
    "quant",
    "anssi",
    "recommandation",
    "préconise",
    "préconisé",
    "recommande",
    "objectif",
    "concernant",
    "mesure",
    "guide",
    "rapport",
}


def mots_cles_cluster(questions, labels, cluster_id, top_n=10):
    textes = questions[labels == cluster_id]
    tfidf = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        stop_words=list(STOP_WORDS_FR),
        token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ]{2,}\b",
        sublinear_tf=True,
    )
    mat = tfidf.fit_transform(textes)
    scores = mat.mean(axis=0).A1
    termes = tfidf.get_feature_names_out()
    top = sorted(zip(scores, termes), reverse=True)[:top_n]
    return [t for _, t in top]


for cid in sorted(set(labels)):
    if cid == -1:
        continue
    kw = mots_cles_cluster(questions, labels, cid)
    n = (labels == cid).sum()
    print(f"Cluster {cid:3d} ({n:5d} questions) : {', '.join(kw)}")

Cluster   0 ( 1758 questions) : toe, sont, sécurité, intégrité, attaquant, journaux, données, configuration, utilisateurs, politique
Cluster   1 (  623 questions) : acronyme, signifie, signifie acronyme, désigne, désigne acronyme, sigle, représente, scada, acronyme scada, api
Cluster   2 (17955 questions) : sont, sécurité, recommandations, système, information, accès, type, impose, doivent, administration
Cluster   3 ( 1275 questions) : ssi, pssi, bord, bord ssi, tableaux, sont, tableaux bord, sécurité, tableau, tableau bord
Cluster   4 (71805 questions) : sont, sécurité, données, doivent, système, type, information, accès, texte, service
Cluster   5 ( 3082 questions) : doivent, sécurité, sont, impose, données, correspondant, type, accès, administration, fréquence
Cluster   6 (13544 questions) : systèmes, gestion, sécurité, impose, données, systèmes gestion, accès, utilisation, surveillance, jour


# Questions représentatives par cluster

In [6]:
def questions_representatives(X_umap, questions, labels, cluster_id, top_n=3):
    mask = labels == cluster_id
    centre = X_umap[mask].mean(axis=0)
    dists = np.linalg.norm(X_umap[mask] - centre, axis=1)
    idx_locaux = np.argsort(dists)[:top_n]
    idx_globaux = np.where(mask)[0][idx_locaux]
    return questions[idx_globaux]


for cid in sorted(set(labels)):
    if cid == -1:
        continue
    kw = mots_cles_cluster(questions, labels, cid)
    n = (labels == cid).sum()
    print(f"\n=== Cluster {cid} ({n} questions) — {', '.join(kw[:5])} ===")
    for q in questions_representatives(X_umap, questions, labels, cid):
        print(f"  • {q}")


=== Cluster 0 (1758 questions) — toe, sont, sécurité, intégrité, attaquant ===
  • Quelle règle doit être respectée concernant l’installation physique du système de la TOE selon H10 ?
  • Quel équipement terminal est prévu ou requis pour la TOE ?
  • Les fonctions « métier » de la TOE sont-elles évaluées pour la sécurité ?

=== Cluster 1 (623 questions) — acronyme, signifie, signifie acronyme, désigne, désigne acronyme ===
  • Que désigne l’acronyme COTS ?
  • Que signifie l’acronyme TOR ?
  • Que signifie l’acronyme COTS ?

=== Cluster 2 (17955 questions) — sont, sécurité, recommandations, système, information ===
  • Quelle mesure l’ANSSI recommande-t-elle pour limiter le risque lié à la personnalisation de l’environnement de développement par chaque développeur ?
  • Quelle étape l’ANSSI recommande‑t‑elle de suivre à votre retour pour vérifier l’intégrité de vos équipements ?
  • Quelle mesure l'ANSSI recommande-t-elle pour assurer la communication ?

=== Cluster 3 (1275 questions)

# Matrice Document × Cluster

In [7]:
df["cluster"] = labels

pivot = (
    df[df["cluster"] != -1]
    .groupby(["document_name", "cluster"])
    .size()
    .unstack(fill_value=0)
)


pivot_norm = pd.DataFrame(
    sk_normalize(pivot.values, norm="l1"),
    index=pivot.index,
    columns=pivot.columns,
)
print(pivot_norm.shape)
pivot_norm.head()

C:\Users\pleroy\AppData\Local\Temp\ipykernel_36172\2297471371.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["cluster"] = labels


(308, 7)


cluster,0,1,2,3,4,5,6
document_name,,,,,,,
2010-12-03_Guide_externalisation.pdf,0.000000,0.001477,0.118168,0.001477,0.791728,0.036928,0.050222
20150713_NP_ANSSI_SDE_firewall_mid_term_v1.0-en.pdf,0.439490,0.000000,0.038217,0.000000,0.522293,0.000000,0.000000
20150713_NP_ANSSI_SDE_firewall_short_term_v1.0-en.pdf,0.445255,0.000000,0.094891,0.000000,0.459854,0.000000,0.000000
20150713_NP_ANSSI_SDE_plc_mid_term_v1.1-en.pdf,0.476744,0.000000,0.005814,0.000000,0.517442,0.000000,0.000000
20150713_NP_ANSSI_SDE_plc_short_term_v1.1-en.pdf,0.246269,0.000000,0.007463,0.000000,0.746269,0.000000,0.000000


#  Similarité entre documents

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(pivot_norm.values)
df_sim = pd.DataFrame(sim, index=pivot_norm.index, columns=pivot_norm.index)

seuil = 0.3
paires = []
docs = list(pivot_norm.index)
for i in range(len(docs)):
    for j in range(i + 1, len(docs)):
        s = sim[i, j]
        if s >= seuil:
            paires.append((docs[i], docs[j], round(s, 3)))

paires.sort(key=lambda x: -x[2])
df_paires = pd.DataFrame(paires, columns=["doc_a", "doc_b", "similarite"])
print(f"{len(df_paires)} paires de documents similaires (seuil={seuil})")
df_paires.head(20)

44440 paires de documents similaires (seuil=0.3)


,doc_a,doc_b,similarite
0,2010-12-03_Guide_externalisation.pdf,np_securiser_jre_notetech.pdf,1.0
1,20150713_NP_ANSSI_SDE_firewall_short_term_v1.0...,20151005_NP_ANSSI_SDE_4067_PJ1_diode_court_ter...,1.0
2,20150713_NP_ANSSI_SDE_vpn_mid_term_v1.0-en.pdf,20150713_NP_ANSSI_SDE_vpn_short_term_v1.0-en.pdf,1.0
3,20220117_np_anssi_sde_commutateur_moyen_terme_...,20230614_np_anssi_sde_automate_court_terme_v1-...,1.0
4,20220117_np_anssi_sde_pare_feu_moyen_terme_v1-...,20241001_NP_ANSSI_SDE_serveur_scada_moyen_term...,1.0
5,20220516_np_anssi_guide_com_crise_cyber_en.pdf,CERTFR-2024-RFX-006-1.pdf,1.0
6,20220516_np_anssi_guide_com_crise_cyber_en.pdf,CERTFR-2024-RFX-008-1.pdf,1.0
7,20220516_np_anssi_guide_com_crise_cyber_en.pdf,CERTFR-2024-RFX-010-1.pdf,1.0
8,20220516_np_anssi_guide_com_crise_cyber_en.pdf,CERTFR-2025-RFX-002-1.pdf,1.0
9,20220516_np_anssi_guide_com_crise_cyber_en.pdf,CERTFR-2025-RFX-007-1.pdf,1.0


#  Clusters communs entre deux documents

# Questions identiques (similarité sémantique directe)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import numpy as np

SEUIL_SIMILARITE = 0.82

sim_matrix = cos_sim(X)

docs = df["document_name"].values
pages = df["source_numero_page"].values
contents = df["content"].values
clusters = labels

i_idx, j_idx = np.triu_indices(len(df), k=1)

scores = sim_matrix[i_idx, j_idx]
masque = (scores >= SEUIL_SIMILARITE) & (docs[i_idx] != docs[j_idx])

i_sel = i_idx[masque]
j_sel = j_idx[masque]
s_sel = scores[masque]

df_similaires = (
    pd.DataFrame(
        {
            "score": s_sel.round(4),
            "question_a": contents[i_sel],
            "doc_a": docs[i_sel],
            "page_a": pages[i_sel],
            "cluster_a": clusters[i_sel],
            "question_b": contents[j_sel],
            "doc_b": docs[j_sel],
            "page_b": pages[j_sel],
            "cluster_b": clusters[j_sel],
        }
    )
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print(
    f"{len(df_similaires)} paires de questions similaires inter-documents (seuil={SEUIL_SIMILARITE})"
)
df_similaires.head(10)

MemoryError: Unable to allocate 45.4 GiB for an array with shape (110365, 110365) and data type float32

In [ ]:
for _, row in df_similaires.iterrows():
    print(f"[{row['score']:.4f}]")
    print(f"  A (cluster {row['cluster_a']}, p.{int(row['page_a'])}) [{row['doc_a']}]")
    print(f"     {row['question_a']}")
    print(f"  B (cluster {row['cluster_b']}, p.{int(row['page_b'])}) [{row['doc_b']}]")
    print(f"     {row['question_b']}")
    print()

# Recherche par question

In [ ]:
MA_QUESTION = "Comment sécuriser un serveur Windows ?"
TOP_N = 5

scores = (
    cos_sim(X, X[df["content"] == MA_QUESTION])[:, 0]
    if MA_QUESTION in df["content"].values
    else None
)

if scores is None:
    candidats = df["content"].values
    from sklearn.feature_extraction.text import TfidfVectorizer as TV

    tv = TV(
        ngram_range=(1, 2),
        stop_words=list(STOP_WORDS_FR),
        token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{3,}\b",
    )
    mat = tv.fit_transform(list(candidats) + [MA_QUESTION])
    scores = cos_sim(mat[-1], mat[:-1])[0]

top_idx = np.argsort(scores)[::-1][:TOP_N]

print(f"Question : {MA_QUESTION}\n")
for rank, idx in enumerate(top_idx, 1):
    print(
        f"  {rank}. [{scores[idx]:.4f}] (cluster {labels[idx]}, p.{int(df.iloc[idx]['source_numero_page'])}) [{df.iloc[idx]['document_name']}]"
    )
    print(f"     {df.iloc[idx]['content']}")
    print()

In [ ]:
def clusters_communs(doc_a, doc_b, pivot, top_n=5):
    if doc_a not in pivot.index or doc_b not in pivot.index:
        return pd.DataFrame()
    intersection = pd.DataFrame(
        {
            "doc_a": pivot.loc[doc_a],
            "doc_b": pivot.loc[doc_b],
        }
    )
    intersection["min"] = intersection.min(axis=1)
    return (
        intersection[intersection["min"] > 0]
        .sort_values("min", ascending=False)
        .head(top_n)
    )


if df_paires.empty:
    print("Aucune paire similaire — essaie de baisser le seuil.")
else:
    doc_a = df_paires.iloc[0]["doc_a"]
    doc_b = df_paires.iloc[0]["doc_b"]
    print(f"{doc_a}  ↔  {doc_b}")
    display(clusters_communs(doc_a, doc_b, pivot))

# Visualisation UMAP 2D

In [ ]:
import plotly.express as px

reducer_2d = umap.UMAP(
    n_components=2,
    metric="cosine",
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)
X_2d = reducer_2d.fit_transform(X_pca)

label_cluster = {-1: "outliers"}
for cid in sorted(set(labels)):
    if cid == -1:
        continue
    kw = mots_cles_cluster(questions, labels, cid, top_n=3)
    label_cluster[cid] = f"C{cid} — {', '.join(kw)}"

df_plot = pd.DataFrame(
    {
        "x": X_2d[:, 0],
        "y": X_2d[:, 1],
        "cluster": [label_cluster[label] for label in labels],
        "question": df["content"].values,
        "document": df["document_name"].values,
        "page": df["source_numero_page"].values,
    }
)

fig = px.scatter(
    df_plot,
    x="x",
    y="y",
    color="cluster",
    hover_data={
        "x": False,
        "y": False,
        "cluster": True,
        "document": True,
        "page": True,
        "question": True,
    },
    title="UMAP 2D — clusters HDBSCAN",
    width=1200,
    height=800,
)
fig.update_traces(marker=dict(size=6, opacity=0.75))
fig.update_layout(legend=dict(orientation="v", x=1.01))

chemin_html = "../../donnees/umap_clusters.html"
fig.write_html(chemin_html)
print(f"Fichier sauvegardé : {chemin_html}")
fig.show()